# Segundo parcial: análisis de ventas con Pandas
Electiva 1 · Fuente: hoja `datos_ventas.csv` enlazada por el docente.
Ejecutar las celdas en orden. Al final se genera `datos_procesados.csv` para entregar.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Descargar los mismos diez registros de la hoja original del docente.
url = 'https://docs.google.com/spreadsheets/d/12CxPsegeQmV__KAJazoedc30ArHLmQHpjxIlsYR7EZU/export?format=csv&gid=770308113'
df = pd.read_csv(url)
df.to_csv('datos_ventas.csv', index=False)
print('Primeras cinco filas:')
display(df.head())
print('Información general:')
df.info()
print('Valores nulos por columna:')
display(df.isna().sum())

In [ ]:
# Se descartan filas incompletas en columnas distintas del precio.
# Luego se rellenan precios faltantes con la media de los precios válidos.
otras = df.columns.difference(['Precio_Unitario'])
limpio = df.dropna(subset=otras).copy()
media_precio = limpio['Precio_Unitario'].mean()
limpio['Precio_Unitario'] = limpio['Precio_Unitario'].fillna(media_precio)
limpio['Fecha'] = pd.to_datetime(limpio['Fecha'], errors='coerce')
limpio = limpio.dropna(subset=['Fecha'])
limpio['Total_Venta'] = limpio['Cantidad'] * limpio['Precio_Unitario']
print('Datos limpios con total de venta:')
display(limpio)

In [ ]:
enero = limpio.loc[limpio['Fecha'].dt.to_period('M') == '2024-01']
ordenado = limpio.sort_values('Total_Venta', ascending=False)
print('Ventas de enero de 2024:')
display(enero)
print('Ventas ordenadas de mayor a menor por Total_Venta:')
display(ordenado)

In [ ]:
ventas_por_producto = limpio.groupby('Producto')['Total_Venta'].sum().sort_values(ascending=False)
unidades_por_producto = limpio.groupby('Producto')['Cantidad'].sum().sort_values(ascending=False)
print('Total de ventas por producto:')
display(ventas_por_producto)
print('Producto con mayor cantidad de unidades vendidas:', unidades_por_producto.index[0])
ax = ventas_por_producto.plot(kind='bar', color='#247ba0', figsize=(7, 4))
ax.set(title='Total de ventas por producto', xlabel='Producto', ylabel='Total de ventas')
plt.tight_layout()
plt.savefig('ventas_por_producto.png', dpi=160)
plt.show()
limpio.to_csv('datos_procesados.csv', index=False, date_format='%Y-%m-%d')
print('Archivo generado: datos_procesados.csv')